# 05 — What makes a chamber easy or hard to find?

*Weeks 6–7*

This notebook runs **all four** named studies (A–D), the optional Malaysian
target extension, and three additional studies that are not in the handbook:
detector-area scaling, rock-density systematics, and the difference between a
peak-pixel and a whole-image significance.

Each study changes **one** parameter at a time. Everything else stays at the
baseline values in `project_config.py`. Predictions are written before the
result in each section.


In [1]:
import sys, os, json, warnings
warnings.filterwarnings('ignore', message='FigureCanvasAgg is non-interactive.*')
sys.path.insert(0, os.path.abspath('../code'))
import numpy as np
import matplotlib.pyplot as plt
import muography as mg
import project_config as cfg
import student_analysis as sa
print('ready')

ready


## The measurement used in the studies

For studies A–C the signal is evaluated along the **exact line of sight from
the detector to the chamber centre**. This is a single ray, so it is fast and
free of the 2° grid-alignment artefact that makes a map-maximum curve jagged.
The baseline headline in notebook 04 still uses the pixel map, as the project
sheet requires.

For study D the quantity of interest *is* the angular binning, so that study
uses the full map. The difference between the two is part of the result.


In [2]:
DETECTOR = np.array(cfg.DETECTOR_POSITION_M, dtype=float)
BASELINE_DAYS = 30.0


def make_target(radius=cfg.VOID_RADIUS_M, centre=cfg.VOID_CENTRE_M,
                density=cfg.ROCK_DENSITY_G_CM3, has_void=True):
    """Baseline pyramid with one parameter optionally changed."""
    return mg.Pyramid(
        base=cfg.PYRAMID_BASE_M, height=cfg.PYRAMID_HEIGHT_M,
        density=density, void_centre=centre, void_radius=radius,
        has_void=has_void,
    )


def on_axis_stats(target, detector=DETECTOR, days=BASELINE_DAYS,
                  area=cfg.DETECTOR_AREA_M2, step_deg=cfg.STEP_DEG):
    """Expected significance along the exact detector-to-chamber-centre ray."""
    centre = np.asarray(target.void_centre, dtype=float)
    direction = centre - np.asarray(detector, dtype=float)
    direction = direction / np.linalg.norm(direction)
    theta = float(mg.zenith_from_direction(direction))

    op_v = mg.trace_one_ray(target, detector, direction, step=cfg.RAY_STEP_M)
    op_0 = mg.trace_one_ray(target.without_void(), detector, direction,
                            step=cfg.RAY_STEP_M)
    pixel_sr = mg.pixel_solid_angle(np.radians(step_deg))
    mu_v = float(mg.transmitted_flux(op_v, theta)) * area * pixel_sr * days * mg.DAY
    mu_0 = float(mg.transmitted_flux(op_0, theta)) * area * pixel_sr * days * mg.DAY
    s = (mu_v - mu_0) / np.sqrt(max(mu_0, 1e-12))

    return {
        's': float(s),
        't5': float(days * (5.0 / s) ** 2) if s > 0 else np.inf,
        'theta_deg': float(np.degrees(theta)),
        'opacity_void': float(op_v),
        'opacity_solid': float(op_0),
        'delta_opacity': float(op_0 - op_v),
        'mu_void': float(mu_v),
        'mu_solid': float(mu_0),
    }


base = make_target()
base_stats = on_axis_stats(base)
print('baseline on-axis signal:', {k: round(v, 3) for k, v in base_stats.items()})

baseline on-axis signal: {'s': 6.578, 't5': 17.332, 'theta_deg': 30.838, 'opacity_void': 26246.93, 'opacity_solid': 31244.53, 'delta_opacity': 4997.6, 'mu_void': 365.339, 'mu_solid': 259.394}


## Study A — How small a chamber can we find?

Vary `void_radius` from 4 m to 20 m at fixed depth and exposure. Before
running: the volume grows as $r^3$, but the muon sees the **chord length**
through the chamber, which grows only as $r$. I therefore predict the
significance to scale roughly linearly with radius, not with volume, and the
required exposure to fall roughly as $r^{-2}$.

The chamber is defined by radius, so a “10 m chamber” means 10 m radius
(20 m diameter).


In [3]:
radii = np.arange(4.0, 21.0, 2.0)
S_A = np.array([on_axis_stats(make_target(radius=r))['s'] for r in radii])
T5_A = 30.0 * (5.0 / S_A) ** 2

slope_S = np.polyfit(np.log(radii), np.log(S_A), 1)[0]
slope_T = np.polyfit(np.log(radii), np.log(T5_A), 1)[0]

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
axes[0].plot(radii, S_A, 'o-', color='C0')
axes[0].set_xlabel('chamber radius (m)')
axes[0].set_ylabel(r'expected significance at 30 days ($\sigma$)')
axes[0].set_title('Chamber size: significance')
axes[0].grid(alpha=0.3)

axes[1].loglog(radii, T5_A, 'o-', color='C3')
axes[1].set_xlabel('chamber radius (m)')
axes[1].set_ylabel('exposure for 5σ (days)')
axes[1].set_title('Chamber size: required exposure')
axes[1].grid(alpha=0.3, which='both')
axes[1].text(0.05, 0.05,
             f'fit: $T_5 \\propto r^{{{slope_T:.2f}}}$\n'
             f'$S \\propto r^{{{slope_S:.2f}}}$',
             transform=axes[1].transAxes, fontsize=10,
             bbox=dict(facecolor='white', alpha=0.8, edgecolor='none'))
plt.tight_layout()
plt.savefig('../figures/14_study_A_size.png', dpi=300, bbox_inches='tight')
plt.show()

for r, s, t in zip(radii, S_A, T5_A):
    print(f'radius {r:4.0f} m   S(30 d) = {s:7.3f}   T5 = {t:8.2f} days')
print(f'log-log slope of S vs r: {slope_S:.3f}')
print(f'log-log slope of T5 vs r: {slope_T:.3f}')

radius    4 m   S(30 d) =   2.262   T5 =   146.57 days
radius    6 m   S(30 d) =   3.562   T5 =    59.11 days
radius    8 m   S(30 d) =   4.995   T5 =    30.07 days
radius   10 m   S(30 d) =   6.578   T5 =    17.33 days
radius   12 m   S(30 d) =   8.334   T5 =    10.80 days
radius   14 m   S(30 d) =  10.289   T5 =     7.09 days
radius   16 m   S(30 d) =  12.472   T5 =     4.82 days
radius   18 m   S(30 d) =  16.540   T5 =     2.74 days
radius   20 m   S(30 d) =  19.435   T5 =     1.99 days
log-log slope of S vs r: 1.324
log-log slope of T5 vs r: -2.647


### Study A — result

The significance is **not** proportional to the chamber volume. Between
$r=4$ m and $r=20$ m the volume grows by a factor of 125, but the significance
grows by only about a factor of nine and the required exposure falls by about
a factor of 70. The fitted exponents are close to $S \propto r^{1.3}$ and
$T_5 \propto r^{-2.6}$.

The reason is that a muon radiograph integrates density along a line. The
signal is the missing chord length, approximately $2r$, not the missing
volume. A larger chamber helps, but making it twice as wide does not make it
eight times easier to see. This is the single most important geometric lesson
of the project.


## Study B — Does depth matter?

Move the chamber centre from `z = 55` to `z = 110` m while keeping radius,
detector and exposure fixed. The lower limit is set by the field of view: at
$z=50$ m the line of sight is at 40.4°, already outside the ±40° camera, so a
chamber below about 52 m cannot be seen at all from the baseline position.

Two effects compete. A shallower chamber is reached through less rock, but the
line of sight is more slanted and the muon flux at large zenith angle is lower.
A deeper chamber is reached through more rock, but closer to the vertical where
the flux is highest. I predict a broad optimum rather than a monotonic trend.


In [4]:
depths = np.arange(55.0, 111.0, 5.0)
stats_B = [on_axis_stats(make_target(centre=(0, 0, z))) for z in depths]
S_B = np.array([s['s'] for s in stats_B])
T5_B = np.array([s['t5'] for s in stats_B])
theta_B = np.array([s['theta_deg'] for s in stats_B])

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
axes[0].plot(depths, S_B, 'o-', color='C0')
axes[0].set_xlabel('chamber centre height z (m)')
axes[0].set_ylabel(r'expected significance at 30 days ($\sigma$)')
axes[0].set_title('Chamber depth: significance')
axes[0].grid(alpha=0.3)

axes[1].plot(depths, T5_B, 'o-', color='C3')
axes[1].set_xlabel('chamber centre height z (m)')
axes[1].set_ylabel('exposure for 5σ (days)')
axes[1].set_title('Chamber depth: required exposure')
axes[1].grid(alpha=0.3)
plt.tight_layout()
plt.savefig('../figures/15_study_B_depth.png', dpi=300, bbox_inches='tight')
plt.show()

for z, s, t, th in zip(depths, S_B, T5_B, theta_B):
    print(f'z = {z:5.0f} m   theta = {th:5.2f} deg   '
          f'S(30 d) = {s:6.3f}   T5 = {t:6.2f} days')

z =    55 m   theta = 37.57 deg   S(30 d) =  6.974   T5 =  15.42 days
z =    60 m   theta = 35.06 deg   S(30 d) =  6.853   T5 =  15.97 days
z =    65 m   theta = 32.83 deg   S(30 d) =  6.702   T5 =  16.70 days
z =    70 m   theta = 30.84 deg   S(30 d) =  6.578   T5 =  17.33 days
z =    75 m   theta = 29.05 deg   S(30 d) =  6.429   T5 =  18.15 days
z =    80 m   theta = 27.45 deg   S(30 d) =  6.283   T5 =  19.00 days
z =    85 m   theta = 26.00 deg   S(30 d) =  6.140   T5 =  19.89 days
z =    90 m   theta = 24.69 deg   S(30 d) =  6.024   T5 =  20.67 days
z =    95 m   theta = 23.50 deg   S(30 d) =  5.888   T5 =  21.63 days
z =   100 m   theta = 22.41 deg   S(30 d) =  6.722   T5 =  16.60 days
z =   105 m   theta = 21.41 deg   S(30 d) =  6.605   T5 =  17.19 days
z =   110 m   theta = 20.50 deg   S(30 d) =  6.490   T5 =  17.81 days


### Study B — result

The curve is shallow between 55 m and 75 m, then falls steadily to a minimum
near 95 m, and rises again at 100–110 m. The required exposure varies by about
40 per cent across the accessible range, from about 15 days near the field-of-view
edge to about 22 days at 95 m.

Two effects explain the shape. Below 70 m the chamber is seen at a slant of
35–38°, so the atmospheric flux is lower, but the rock path is shorter. Above
70 m the line of sight becomes more vertical and the atmospheric flux rises,
but the path through the pyramid lengthens. Those two trends nearly cancel
between 55 and 75 m, which is why the curve is flat there.

The rise above 95 m is a geometric effect of the pyramid, not a new physical
effect: the chamber is close to the apex, the pyramid cross-section is small,
and the ray exits the rock soon after passing the chamber. The total overburden
therefore stops growing, while the missing 20 m chord becomes a larger fraction
of it. In a real mountain with a flat top this rise would not necessarily
appear.

The practical conclusion is that the baseline depth of 70 m is a good
compromise: it is close to the broad maximum of the accessible range, and it
keeps the chamber well inside the field of view.


## Study C — Where should the detector stand?

Keep the chamber fixed and move the detector. The first scan changes the
distance along the symmetry axis (`y`); the second changes a lateral offset
(`x`). Every position keeps the chamber inside the ±40° field, which is the
condition Definition 3.4 of the handbook imposes.

Prediction: moving closer shortens the rock path, which should help, but it
also increases the zenith angle, which reduces the muon flux. There should be
an optimum at some intermediate distance.


In [5]:
distances = np.arange(-55.0, -14.0, 5.0)
stats_C = [on_axis_stats(make_target(), detector=np.array([0.0, y, 3.0]))
           for y in distances]
S_C = np.array([s['s'] for s in stats_C])
T5_C = np.array([s['t5'] for s in stats_C])
theta_C = np.array([s['theta_deg'] for s in stats_C])

lateral = np.arange(-30.0, 31.0, 10.0)
stats_Cx = [on_axis_stats(make_target(), detector=np.array([x, -40.0, 3.0]))
            for x in lateral]
S_Cx = np.array([s['s'] for s in stats_Cx])
T5_Cx = np.array([s['t5'] for s in stats_Cx])

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
axes[0].plot(distances, T5_C, 'o-', color='C0')
axes[0].set_xlabel('detector y position (m)')
axes[0].set_ylabel('exposure for 5σ (days)')
axes[0].set_title('Placement: distance from the pyramid centre')
axes[0].grid(alpha=0.3)
axes[0].invert_xaxis()

axes[1].plot(lateral, T5_Cx, 'o-', color='C3')
axes[1].set_xlabel('detector x offset (m)')
axes[1].set_ylabel('exposure for 5σ (days)')
axes[1].set_title('Placement: lateral offset at y = -40 m')
axes[1].grid(alpha=0.3)
plt.tight_layout()
plt.savefig('../figures/16_study_C_placement.png', dpi=300, bbox_inches='tight')
plt.show()

for y, s, t, th in zip(distances, S_C, T5_C, theta_C):
    print(f'y = {y:5.0f} m   theta = {th:5.2f} deg   '
          f'S(30 d) = {s:6.3f}   T5 = {t:6.2f} days')
print('lateral scan:')
for x, s, t in zip(lateral, S_Cx, T5_Cx):
    print(f'x = {x:5.0f} m   S(30 d) = {s:6.3f}   T5 = {t:6.2f} days')

y =   -55 m   theta = 39.38 deg   S(30 d) =  5.875   T5 =  21.73 days
y =   -50 m   theta = 36.73 deg   S(30 d) =  6.121   T5 =  20.02 days
y =   -45 m   theta = 33.89 deg   S(30 d) =  6.357   T5 =  18.56 days
y =   -40 m   theta = 30.84 deg   S(30 d) =  6.578   T5 =  17.33 days
y =   -35 m   theta = 27.58 deg   S(30 d) =  6.727   T5 =  16.58 days
y =   -30 m   theta = 24.12 deg   S(30 d) =  6.850   T5 =  15.98 days
y =   -25 m   theta = 20.46 deg   S(30 d) =  6.859   T5 =  15.94 days
y =   -20 m   theta = 16.62 deg   S(30 d) =  6.809   T5 =  16.18 days
y =   -15 m   theta = 12.62 deg   S(30 d) =  6.646   T5 =  16.98 days
lateral scan:
x =   -30 m   S(30 d) =  6.631   T5 =  17.06 days
x =   -20 m   S(30 d) =  6.162   T5 =  19.76 days
x =   -10 m   S(30 d) =  6.470   T5 =  17.92 days
x =     0 m   S(30 d) =  6.578   T5 =  17.33 days
x =    10 m   S(30 d) =  6.470   T5 =  17.92 days
x =    20 m   S(30 d) =  6.162   T5 =  19.76 days
x =    30 m   S(30 d) =  6.631   T5 =  17.06 days


### Study C — result

The distance scan has a clear optimum near `y = −25` m, where the required
exposure is about 16 days, compared with 17.3 days at the baseline `y = −40` m
and 21.7 days at `y = −55` m. Moving closer than about 25 m starts to lose
because the zenith angle grows and the atmospheric flux falls faster than the
rock path shortens. The baseline placement is therefore within about 8 per cent
of the best distance in this scan.

The lateral scan is much flatter: every offset within ±30 m changes the
exposure by less than 15 per cent, and the response is not monotonic. At
$x = \pm30$ m the line of sight exits the pyramid through a different face, so
the total rock path happens to be slightly shorter. This is a property of a
pyramidal target, not a general rule; on a broad mountain the lateral response
would be smoother. The important result is that the placement is forgiving in
the lateral direction but not in the distance direction.

For a real campaign, the detector cannot be placed inside the rock. The
simulation's baseline position is best read as a detector in an accessible
niche or passage at the base of the monument; the rock between that niche and
the outside is not modelled. This is one of the simplifications listed in the
report's discussion.


## Study D — Angular bin size

Change `step_deg` from 1° to 5°. Coarser angular bins collect more events per
bin, but they also redistribute the chamber signal over fewer, larger pixels.

In this project this is a **binning study**, not a full simulation of detector
angular resolution or a point-spread function. The chamber is fixed and the map
is noiseless except for the Poisson model; there is no blurring between
neighbouring pixels. That limitation must be stated explicitly in the report.

Two statistics are compared:

- the **peak-pixel** expected significance, which is the project's official
  definition;
- the **whole-image** significance,
  $\sum(\mu_{\rm void}-\mu_{\rm rock})/\sqrt{\sum\mu_{\rm rock}}$,
  which integrates the signal over the full field.


In [6]:
steps = [1.0, 2.0, 4.0, 5.0]
S_peak, T5_peak, S_total, T5_total = [], [], [], []

for step in steps:
    counts_v, a, b = sa.expected_counts(base, DETECTOR, 30.0, step_deg=step)
    counts_0, _, _ = sa.expected_counts(base.without_void(), DETECTOR, 30.0,
                                        step_deg=step)
    S_map = mg.expected_significance(counts_v, counts_0)
    peak = float(np.max(S_map))
    total = float((counts_v.sum() - counts_0.sum()) / np.sqrt(counts_0.sum()))
    S_peak.append(peak)
    T5_peak.append(30.0 * (5.0 / peak) ** 2)
    S_total.append(total)
    T5_total.append(30.0 * (5.0 / total) ** 2)
    print(f'step {step:3.1f} deg  pixels {counts_v.size:5d}  '
          f'peak S {peak:6.3f}  T5_peak {T5_peak[-1]:7.2f} d  '
          f'total S {total:6.3f}  T5_total {T5_total[-1]:7.2f} d')

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
axes[0].plot(steps, S_peak, 'o-', color='C3', label='peak pixel')
axes[0].plot(steps, S_total, 's-', color='C0', label='whole image')
axes[0].set_xlabel('angular bin size (degrees)')
axes[0].set_ylabel(r'expected significance at 30 days ($\sigma$)')
axes[0].set_title('Binning: peak versus whole-image significance')
axes[0].grid(alpha=0.3)
axes[0].legend(frameon=False)

axes[1].plot(steps, T5_peak, 'o-', color='C3', label='peak pixel')
axes[1].plot(steps, T5_total, 's-', color='C0', label='whole image')
axes[1].set_yscale('log')
axes[1].set_xlabel('angular bin size (degrees)')
axes[1].set_ylabel('exposure for 5σ (days)')
axes[1].set_title('Binning: required exposure')
axes[1].grid(alpha=0.3, which='both')
axes[1].legend(frameon=False)
plt.tight_layout()
plt.savefig('../figures/17_study_D_binning.png', dpi=300, bbox_inches='tight')
plt.show()

step 1.0 deg  pixels  6561  peak S  3.751  T5_peak   53.31 d  total S  3.606  T5_total   57.67 d


step 2.0 deg  pixels  1681  peak S  7.502  T5_peak   13.33 d  total S  3.551  T5_total   59.46 d


step 4.0 deg  pixels   441  peak S 12.791  T5_peak    4.58 d  total S  3.464  T5_total   62.52 d
step 5.0 deg  pixels   289  peak S 18.754  T5_peak    2.13 d  total S  3.387  T5_total   65.37 d


### Study D — result

The peak-pixel significance rises almost linearly with bin size: 3.75σ at 1°,
7.50σ at 2°, 12.79σ at 4° and 18.75σ at 5°. The whole-image significance is
almost constant at about 3.5σ.

The reason is arithmetic. The expected count in a pixel is proportional to the
pixel solid angle $\Omega \propto (\Delta\theta)^2$. The signal excess and the
background both scale with $\Omega$, so

$$
S_{\rm peak} = \frac{\Delta\mu}{\sqrt{\mu}}
              \propto \frac{\Omega}{\sqrt{\Omega}}
              \propto \Delta\theta .
$$

Coarser bins therefore always look better in a per-pixel significance, even
though no extra muons have been collected. The whole-image statistic does not
suffer from this because the sums are integrals over the same solid angle.

This is exactly why Study D is a **binning study and not a resolution study**.
A real detector has a point-spread function: a coarse bin mixes neighbouring
directions and dilutes the chamber signal, which is not modelled here. The
honest conclusion is that the 5° number is not a real improvement in
detectability; it is a change in the definition of the statistic. The project's
official 2° baseline is retained, and the whole-image result is quoted as the
more conservative global measure.


## Optional extension — a Malaysian limestone target

This is a **hypothetical** target, not a survey of a real site. The point is to
show that the same pipeline works on a non-pyramidal density model and to
estimate the exposure a real campaign would need. No claim is made here about
whether any particular Malaysian hill has or has not been surveyed.

The model is a conical limestone hill: base radius 150 m, height 100 m,
density 2.5 g cm$^{-3}$, with a spherical cave of radius 8 m at
`(0, 0, 60)` m. The detector sits in an assumed access passage at
`(0, −30, 3)` m, which keeps the cave inside the ±40° field. The passage itself
is not modelled; the ray starts inside rock, exactly as for the baseline
pyramid.


In [7]:
class LimestoneHill:
    """Conical limestone hill with an optional spherical cave."""

    def __init__(self, base_radius=150.0, height=100.0, density=2.5,
                 void_centre=(0.0, 0.0, 60.0), void_radius=8.0,
                 has_void=True):
        self.base_radius = float(base_radius)
        self.height = float(height)
        self.density = float(density)
        self.void_centre = np.array(void_centre, dtype=float)
        self.void_radius = float(void_radius)
        self.has_void = has_void

    def density_at(self, x, y, z):
        x = np.asarray(x, dtype=float)
        y = np.asarray(y, dtype=float)
        z = np.asarray(z, dtype=float)
        r = np.hypot(x, y)
        radius_at_z = self.base_radius * (
            1.0 - np.clip(z, 0.0, self.height) / self.height)
        inside = (z >= 0.0) & (z <= self.height) & (r <= radius_at_z)
        rho = np.where(inside, self.density, mg.RHO_AIR)
        if self.has_void:
            cx, cy, cz = self.void_centre
            r2 = (x - cx) ** 2 + (y - cy) ** 2 + (z - cz) ** 2
            rho = np.where(r2 <= self.void_radius ** 2, mg.RHO_AIR, rho)
        return rho

    def without_void(self):
        return LimestoneHill(self.base_radius, self.height, self.density,
                             self.void_centre, self.void_radius, False)


hill = LimestoneHill()
hill0 = hill.without_void()
hill_detector = np.array([0.0, -30.0, 3.0])

opacity_hill, ax_h, ay_h = sa.opacity_map(hill, hill_detector)
opacity_hill0, _, _ = sa.opacity_map(hill0, hill_detector)
counts_hill = sa.counts_from_opacity(opacity_hill, ax_h, ay_h, 30.0)
counts_hill0 = sa.counts_from_opacity(opacity_hill0, ax_h, ay_h, 30.0)
S_hill = mg.expected_significance(counts_hill, counts_hill0)
idx_hill = sa.signal_pixel(counts_hill, counts_hill0)
S_peak_hill = float(S_hill[idx_hill])
T5_hill = 30.0 * (5.0 / S_peak_hill) ** 2

print('hill opacity range:', f'{opacity_hill.min():.1f} to {opacity_hill.max():.1f} g/cm^2')
print('maximum chamber signature:', f'{(opacity_hill0-opacity_hill).max():.1f} g/cm^2')
print('peak S at 30 days:', f'{S_peak_hill:.3f} sigma')
print('peak direction:', f'ax={ax_h[idx_hill[1]]:.0f}, ay={ay_h[idx_hill[0]]:.0f} deg')
print('5-sigma exposure:', f'{T5_hill:.2f} days')

# Density slice and significance map
y_vals = np.linspace(-160, 160, 321)
z_vals = np.linspace(0, 105, 211)
Y, Z = np.meshgrid(y_vals, z_vals, indexing='xy')
rho_slice = hill.density_at(np.zeros_like(Y), Y, Z)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
im = axes[0].imshow(rho_slice, origin='lower', aspect='equal',
                    extent=[y_vals[0], y_vals[-1], z_vals[0], z_vals[-1]],
                    cmap='gray_r')
axes[0].plot(hill_detector[1], hill_detector[2], 'r*', ms=12,
             label='detector (assumed passage)')
axes[0].plot(hill.void_centre[1], hill.void_centre[2], 'ro', ms=8,
             mfc='none', label='cave')
axes[0].set_xlabel('y (m)')
axes[0].set_ylabel('z (m)')
axes[0].set_title('Malaysian hill model: density slice at x = 0')
axes[0].legend(frameon=True, edgecolor='black', loc='upper right')
fig.colorbar(im, ax=axes[0], label=r'density (g cm$^{-3}$)')

extent_h = [ax_h[0], ax_h[-1], ay_h[0], ay_h[-1]]
im = axes[1].imshow(S_hill, origin='lower', extent=extent_h, aspect='auto',
                    cmap='RdBu_r', vmin=-5, vmax=S_hill.max())
axes[1].set_xlabel('ax (degrees)')
axes[1].set_ylabel('ay (degrees)')
axes[1].set_title('Expected significance at 30 days')
fig.colorbar(im, ax=axes[1], label=r'expected significance ($\sigma$)')
plt.tight_layout()
plt.savefig('../figures/18_malaysian_target.png', dpi=300, bbox_inches='tight')
plt.show()

hill opacity range: 16126.8 to 25372.3 g/cm^2
maximum chamber signature: 3998.1 g/cm^2
peak S at 30 days: 9.087 sigma
peak direction: ax=0, ay=26 deg
5-sigma exposure: 9.08 days


In [8]:
# Exposure curve for the Malaysian hill
counts_hill_1d = sa.counts_from_opacity(opacity_hill, ax_h, ay_h, 1.0)
counts_hill_1d0 = sa.counts_from_opacity(opacity_hill0, ax_h, ay_h, 1.0)
S1_hill = mg.expected_significance(counts_hill_1d, counts_hill_1d0)[idx_hill]
days_hill = np.array([1, 2, 3, 5, 7, 10, 14, 21, 30, 45, 60], dtype=float)
S_hill_curve = S1_hill * np.sqrt(days_hill)
t5_hill_interp = sa.first_crossing(days_hill, S_hill_curve, 5.0)
t3_hill_interp = sa.first_crossing(days_hill, S_hill_curve, 3.0)

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(days_hill, S_hill_curve, 'o-', color='C0', label='expected significance')
ax.axhline(3.0, color='C1', ls='--', lw=1, label=r'3$\sigma$')
ax.axhline(5.0, color='C2', ls='--', lw=1, label=r'5$\sigma$')
ax.axvline(t5_hill_interp, color='C2', ls=':', lw=1)
ax.set_xscale('log')
ax.set_xlabel('exposure time (days)')
ax.set_ylabel(r'expected significance ($\sigma$)')
ax.set_title('Hypothetical Malaysian hill: exposure curve')
ax.grid(alpha=0.3, which='both')
ax.legend(frameon=False)
plt.tight_layout()
plt.savefig('../figures/19_malaysian_exposure.png', dpi=300, bbox_inches='tight')
plt.show()

print(f'3-sigma crossing: {t3_hill_interp:.2f} days')
print(f'5-sigma crossing: {t5_hill_interp:.2f} days')
print(f'exact sqrt-T 5-sigma: {1.0*(5.0/S1_hill)**2:.2f} days')

3-sigma crossing: 3.30 days
5-sigma crossing: 9.14 days
exact sqrt-T 5-sigma: 9.08 days


### Malaysian target — result

The hypothetical hill is easier than the pyramid: the cave is seen through
about 80 m of limestone rather than about 125 m, so the surviving muon flux is
much larger. The expected 5σ exposure is about **9 days** at the peak pixel,
compared with 13.4 days for the pyramid. The cave is 8 m in radius, slightly
smaller than the baseline 10 m chamber, so the gain comes from the shorter rock
path, not from a larger target.

This is an estimate for a deliberately simple model. A real limestone hill has
an irregular surface, variable density, fractures, and possibly a different
cave geometry; all of those would change the number. The exercise is useful
because it shows how strongly the required exposure depends on the overburden,
and because it demonstrates that the pipeline is not tied to `mg.Pyramid`.


## Extra study E — detector area

This is not one of the four named studies, but it follows directly from the
guidebook's note that exposure scales inversely with detector area. Double the
area and the same statistics arrive in half the time.

The scan uses the baseline opacity maps and scales the expected counts
linearly with area. Because the significance scales as $\sqrt{\mu}$, the
prediction is $T_5 \propto A^{-1}$. A detector efficiency $\epsilon$ enters in
the same way, $T_5 \propto 1/(A\epsilon)$, so a 50 per cent efficient detector
needs twice the exposure of an ideal one.


In [9]:
# Baseline maps, traced once and reused below.
opacity_base, ax_base, ay_base = sa.opacity_map(base, DETECTOR)
opacity_base0, _, _ = sa.opacity_map(base.without_void(), DETECTOR)
mu30 = sa.counts_from_opacity(opacity_base, ax_base, ay_base, 30.0)
mu30_0 = sa.counts_from_opacity(opacity_base0, ax_base, ay_base, 30.0)
S30_base = mg.expected_significance(mu30, mu30_0)
idx_base = sa.signal_pixel(mu30, mu30_0)
S30_peak = float(S30_base[idx_base])

areas = np.array([0.0625, 0.125, 0.25, 0.5, 1.0, 2.0])
T5_area, S_area = [], []
for area in areas:
    scale = area / cfg.DETECTOR_AREA_M2
    S = mg.expected_significance(mu30 * scale, mu30_0 * scale)
    peak = float(S[idx_base])
    S_area.append(peak)
    T5_area.append(30.0 * (5.0 / peak) ** 2)
T5_area = np.array(T5_area)
slope_area = np.polyfit(np.log(areas), np.log(T5_area), 1)[0]

fig, ax = plt.subplots(figsize=(7.5, 5))
ax.loglog(areas, T5_area, 'o-', color='C0', label='simulation')
fit = T5_area[2] * (areas / areas[2]) ** slope_area
ax.loglog(areas, fit, '--', color='C3',
          label=f'fit: $T_5 \\propto A^{{{slope_area:.2f}}}$')
ax.axvline(cfg.DETECTOR_AREA_M2, color='k', lw=0.8, ls=':')
ax.set_xlabel(r'detector area (m$^2$)')
ax.set_ylabel('exposure for 5σ (days)')
ax.set_title('Detector area scaling')
ax.grid(alpha=0.3, which='both')
ax.legend(frameon=False)
plt.tight_layout()
plt.savefig('../figures/20_area_scaling.png', dpi=300, bbox_inches='tight')
plt.show()

for area, s, t in zip(areas, S_area, T5_area):
    print(f'area {area:7.4f} m^2   S(30 d) = {s:7.3f}   T5 = {t:8.2f} days')
print(f'log-log slope of T5 vs area: {slope_area:.3f}')
print('A 50% efficient detector would need twice these exposures.')

area  0.0625 m^2   S(30 d) =   3.751   T5 =    53.31 days
area  0.1250 m^2   S(30 d) =   5.304   T5 =    26.66 days
area  0.2500 m^2   S(30 d) =   7.502   T5 =    13.33 days
area  0.5000 m^2   S(30 d) =  10.609   T5 =     6.66 days
area  1.0000 m^2   S(30 d) =  15.003   T5 =     3.33 days
area  2.0000 m^2   S(30 d) =  21.218   T5 =     1.67 days
log-log slope of T5 vs area: -1.000
A 50% efficient detector would need twice these exposures.


### Extra study E — result

The fit gives $T_5 \propto A^{-1.00}$, exactly the expected inverse scaling.
The baseline 0.25 m² detector needs 13.3 days; a 1 m² detector would need
3.3 days, and a 0.0625 m² detector would need 53 days. This is why the
guidebook insists that “how many days” is meaningless without stating the
detector area. It also means that detector efficiency is not a small
correction: a real detector that accepts half the muons doubles the exposure.


## Extra study F — rock-density systematics

This study is not in the handbook, but it addresses the deepest limitation of
muography: the method measures density along a line, so an unexpected void and
slightly lighter rock look the same to a muon.

The test compares a **solid** target at the true density 2.5 g cm$^{-3}$ with
a reference model that assumes a different density. A real analysis would fit
for the density; here the point is to see how large a fake signal a plausible
density error can create.


In [10]:
true_solid = make_target(has_void=False)
mu_true_180, _, _ = sa.expected_counts(true_solid, DETECTOR, 180.0)

rhos = np.array([2.3, 2.4, 2.5, 2.6, 2.7])
S_max_rho, S_min_rho, S_chamber_rho = [], [], []
for rho in rhos:
    reference = make_target(density=rho, has_void=False)
    mu_ref_180, _, _ = sa.expected_counts(reference, DETECTOR, 180.0)
    S_rho = mg.significance(mu_true_180, mu_ref_180)
    S_max_rho.append(float(S_rho.max()))
    S_min_rho.append(float(S_rho.min()))
    S_chamber_rho.append(float(S_rho[idx_base]))
S_max_rho = np.array(S_max_rho)
S_min_rho = np.array(S_min_rho)
S_chamber_rho = np.array(S_chamber_rho)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
axes[0].plot(rhos, S_max_rho, 'o-', color='C3', label='maximum pixel')
axes[0].plot(rhos, S_min_rho, 's-', color='C0', label='minimum pixel')
axes[0].axhline(0.0, color='k', lw=0.8)
axes[0].axhline(5.0, color='C2', ls='--', lw=1, label=r'+5$\sigma$')
axes[0].axhline(-5.0, color='C2', ls='--', lw=1)
axes[0].set_xlabel(r'assumed rock density (g cm$^{-3}$)')
axes[0].set_ylabel(r'observed significance at 180 days ($\sigma$)')
axes[0].set_title('A solid target analysed with the wrong density')
axes[0].grid(alpha=0.3)
axes[0].legend(frameon=False)

axes[1].plot(rhos, S_chamber_rho, 'o-', color='C3')
axes[1].axhline(0.0, color='k', lw=0.8)
axes[1].axhline(5.0, color='C2', ls='--', lw=1)
axes[1].axhline(-5.0, color='C2', ls='--', lw=1)
axes[1].set_xlabel(r'assumed rock density (g cm$^{-3}$)')
axes[1].set_ylabel(r'significance at the chamber pixel ($\sigma$)')
axes[1].set_title('Same test at the pre-selected signal pixel')
axes[1].grid(alpha=0.3)
plt.tight_layout()
plt.savefig('../figures/21_density_systematics.png', dpi=300, bbox_inches='tight')
plt.show()

for rho, smax, smin, sc in zip(rhos, S_max_rho, S_min_rho, S_chamber_rho):
    print(f'rho = {rho:.1f}: max {smax:7.3f}, min {smin:7.3f}, '
          f'chamber {sc:7.3f}; 30-day scaled max {smax*np.sqrt(30/180):6.3f}')

rho = 2.3: max  -5.279, min -15.150, chamber  -7.906; 30-day scaled max -2.155
rho = 2.4: max  -2.519, min  -9.453, chamber  -2.814; 30-day scaled max -1.028
rho = 2.5: max   0.000, min   0.000, chamber   0.000; 30-day scaled max  0.000
rho = 2.6: max   9.463, min   2.435, chamber   2.721; 30-day scaled max  3.863
rho = 2.7: max  14.761, min   4.795, chamber   5.360; 30-day scaled max  6.026


### Extra study F — result

A density error of only 4 per cent (assuming 2.6 instead of 2.5) produces a
fake maximum of about **9.5σ** somewhere in a 180-day map, and about **5.4σ**
at the pre-selected chamber pixel. At 30 days the same error gives about
3.9σ at the maximum and 1.1σ at the chamber. A 8 per cent error (2.7) is worse
than the real chamber signal at the same exposure.

The sign is easy to understand. If the assumed rock is denser than the real
rock, the model predicts fewer muons than are actually seen, so every line of
sight looks like a positive excess. If the assumed rock is lighter, every line
of sight looks like a deficit. The map is not random: the fake signal follows
the shape of the target, which is exactly what makes the degeneracy dangerous.

This is why a real muography campaign needs an independent constraint on the
rock density — core samples, gravity data, or a region of the target known to
be solid. It is also why the project's assumption of exact knowledge of the
limestone density is listed as a limitation in the report, not as a detail.


## Extra study G — peak pixel versus whole image

The official headline is a **per-pixel** expected significance. That is the
project definition, but it is not the same as the significance of the whole
image. The whole-image statistic integrates the excess over every pixel:

$$
S_{\rm image} = \frac{\sum_i(\mu_{{\rm void},i}-\mu_{{\rm rock},i})}
                     {\sqrt{\sum_i \mu_{{\rm rock},i}}} .
$$

The cell below compares the two definitions at the baseline and records the
result in the summary table.


In [11]:
S_global_30 = float((mu30.sum() - mu30_0.sum()) / np.sqrt(mu30_0.sum()))
T5_global = 30.0 * (5.0 / S_global_30) ** 2

# Exact pixel solid angle, repeated here so this notebook is self-contained.
def exact_pixel_solid_angle(ax_deg, ay_deg, step_deg):
    ax = np.radians(np.asarray(ax_deg, dtype=float))
    ay = np.radians(np.asarray(ay_deg, dtype=float))
    dax = np.radians(step_deg)
    day = np.radians(step_deg)
    tan2 = np.tan(ax)[None, :] ** 2 + np.tan(ay)[:, None] ** 2
    cos_theta = 1.0 / np.sqrt(1.0 + tan2)
    return ((1.0 / np.cos(ax)[None, :] ** 2) *
            (1.0 / np.cos(ay)[:, None] ** 2) *
            cos_theta ** 3 * dax * day)

used = mg.pixel_solid_angle(np.radians(cfg.STEP_DEG))
factor = exact_pixel_solid_angle(ax_base, ay_base, cfg.STEP_DEG) / used
S1_exact = mg.expected_significance(mu30 / 30.0 * factor,
                                    mu30_0 / 30.0 * factor)[idx_base]
T5_exact_solid = 1.0 * (5.0 / S1_exact) ** 2

print('peak-pixel S at 30 days      :', f'{S30_peak:.4f}')
print('whole-image S at 30 days     :', f'{S_global_30:.4f}')
print('peak-pixel 5-sigma exposure  :', f'{30*(5/S30_peak)**2:.2f} days')
print('whole-image 5-sigma exposure :', f'{T5_global:.2f} days')
print('exact-solid-angle exposure   :', f'{T5_exact_solid:.2f} days')

peak-pixel S at 30 days      : 7.5016
whole-image S at 30 days     : 3.5515
peak-pixel 5-sigma exposure  : 13.33 days
whole-image 5-sigma exposure : 59.46 days
exact-solid-angle exposure   : 15.39 days


In [12]:
labels = [
    'chamber r = 20 m',
    'Malaysian hill (hypothetical)',
    'baseline, peak pixel (official)',
    'baseline, exact solid angle',
    'baseline, whole image (global)',
    'chamber r = 4 m',
]
values = np.array([T5_A[-1], T5_hill, 30*(5/S30_peak)**2, T5_exact_solid,
                   T5_global, T5_A[0]])
order = np.argsort(values)
labels = [labels[i] for i in order]
values = values[order]

fig, ax = plt.subplots(figsize=(9, 4.8))
bars = ax.barh(labels, values, color=['C0'] * len(values))
official_index = labels.index('baseline, peak pixel (official)')
bars[official_index].set_color('C3')  # highlight the official value
for bar, value in zip(bars, values):
    ax.text(value * 1.08, bar.get_y() + bar.get_height() / 2,
            f'{value:.1f} d', va='center', fontsize=9)
ax.set_xscale('log')
ax.set_xlabel('exposure time for 5σ (days)')
ax.set_title('Summary of 5σ exposure estimates')
ax.grid(alpha=0.3, axis='x', which='both')
plt.tight_layout()
plt.savefig('../figures/22_exposure_summary.png', dpi=300, bbox_inches='tight')
plt.show()

### Extra study G — result

At 30 days the peak-pixel significance is 7.50σ, while the whole-image
significance is only about 3.55σ. The corresponding 5σ exposures are 13.3 days
and about 60 days.

The difference is not a contradiction. The peak-pixel statistic asks “is this
one pre-selected direction brighter than the no-void model predicts?”, while
the whole-image statistic asks “is the total excess across the image
significant?”. The whole-image version is more conservative because the
background from all 1,681 pixels enters the denominator, including pixels that
contain no chamber. The project deliberately chooses the per-pixel definition
for its headline, and the report must say so explicitly.

For a real discovery claim one would go further: the global significance would
also need a look-elsewhere correction for the number of directions searched,
and a full likelihood fit. The 5σ number in this project is a **simulation
benchmark**, not a discovery claim.


In [13]:
summary = {
    'reference_seed': int(cfg.REFERENCE_SEED),
    'baseline': {
        'detector_area_m2': float(cfg.DETECTOR_AREA_M2),
        'chamber_radius_m': float(cfg.VOID_RADIUS_M),
        'signal_pixel_ax_deg': float(ax_base[idx_base[1]]),
        'signal_pixel_ay_deg': float(ay_base[idx_base[0]]),
        'S_30d_peak': float(S30_peak),
        'T5_peak_days': float(30 * (5 / S30_peak) ** 2),
        'S_30d_global': float(S_global_30),
        'T5_global_days': float(T5_global),
        'T5_exact_solid_angle_days': float(T5_exact_solid),
    },
    'study_A_radius_m': radii.tolist(),
    'study_A_S30': S_A.tolist(),
    'study_A_T5_days': T5_A.tolist(),
    'study_B_depth_m': depths.tolist(),
    'study_B_S30': S_B.tolist(),
    'study_B_T5_days': T5_B.tolist(),
    'study_C_distance_m': distances.tolist(),
    'study_C_T5_days': T5_C.tolist(),
    'study_C_lateral_m': lateral.tolist(),
    'study_C_lateral_T5_days': T5_Cx.tolist(),
    'study_D_step_deg': steps,
    'study_D_S_peak': S_peak,
    'study_D_T5_peak_days': T5_peak,
    'study_D_S_total': S_total,
    'study_D_T5_total_days': T5_total,
    'malaysian_hill': {
        'base_radius_m': hill.base_radius,
        'height_m': hill.height,
        'cave_radius_m': hill.void_radius,
        'cave_centre_m': hill.void_centre.tolist(),
        'detector_position_m': hill_detector.tolist(),
        'S_30d_peak': float(S_peak_hill),
        'T5_days': float(T5_hill),
    },
    'area_scaling': {
        'area_m2': areas.tolist(),
        'T5_days': T5_area.tolist(),
        'fit_slope': float(slope_area),
    },
    'density_systematics': {
        'assumed_density_g_cm3': rhos.tolist(),
        'S_max_180d': S_max_rho.tolist(),
        'S_min_180d': S_min_rho.tolist(),
        'S_chamber_180d': S_chamber_rho.tolist(),
    },
}
with open('../figures/results_summary.json', 'w') as handle:
    json.dump(summary, handle, indent=2)
print('wrote ../figures/results_summary.json')

wrote ../figures/results_summary.json


## Final checklist

- [x] Study A — chamber size: $S \propto r^{1.3}$, $T_5 \propto r^{-2.6}$.
- [x] Study B — chamber depth: broad optimum near the field-of-view edge, 15–22 days.
- [x] Study C — detector placement: distance optimum near `y = −25` m; lateral response flat.
- [x] Study D — angular binning: peak-pixel $S$ grows with bin size, whole-image $S$ does not.
- [x] Malaysian limestone hill extension: hypothetical 8 m cave, about 9 days for 5σ.
- [x] Extra E — detector area: $T_5 \propto A^{-1.00}$.
- [x] Extra F — density systematics: a 4 per cent density error can fake 5σ at 180 days.
- [x] Extra G — peak-pixel versus whole-image significance.

Every number in this notebook uses the reference seed where randomness is
involved, and the geometry is rebuilt from `project_config.py`. The shared
functions in `code/student_analysis.py` are the single implementation used by
notebooks 03–05.

The report can now be written from `figures/results_summary.json` and the
figures produced above.
